# Session 1 — Python preliminaries & the eye-tracking data landscape  
**Duration:** 1.5 hours · NBML analysis training

### Learning goals
1. Survive in a Jupyter notebook (run cells, read errors, restart kernel).
2. Load Tobii and Pupil Labs tables with **pandas**.
3. Know which famous Python libraries we will use this week — and what each is for.
4. Map **files → analysis questions** for the workshop datasets.

> Audience assumption: little or no Python. We go slowly and compare every step to Excel.


## 0. Classroom setup (10 min)

1. Open a terminal in the course repo root.
2. Activate your environment (see `workshop/SETUP.md`).
3. Launch Jupyter:

```bash
jupyter notebook workshop/sessions
```

4. Open this file (`S01_...`).

**Rule of the room:** if a cell fails, read the **last line** of the error first.


## 1. Variables, lists, and a tiny loop (15 min)

In [ ]:
participant = "Recording3"
aois = ["food-pic", "buy", "not-buy"]
print("Hello,", participant)
for i, name in enumerate(aois, start=1):
    print(i, name)

## 2. Libraries we will actually use (15 min)

| Library | Why eye-tracking people care |
|---------|------------------------------|
| **numpy** | Fast numeric arrays (gaze samples, velocity) |
| **pandas** | Tables — like Excel sheets with scripts |
| **matplotlib** / **seaborn** | Publication-style figures |
| **scipy** | Stats utilities, filters, t-tests |
| **statsmodels** | Regression / ANOVA with academic output |
| **pingouin** | Friendly stats API (t-tests, corr, reliability) |
| **Pillow (PIL)** | Load stimulus images for overlays |
| **openpyxl** | Read Tobii `.xlsx` exports |

Also know the names (we may only demo briefly): **neurokit2** (GSR/EDA), **PyGaze** / analysis tools, vendor SDKs (Tobii, Pupil).  
We prioritize **open tables + transparent code** over black-box GUIs.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0])
print("numpy", np.__version__, "| pandas", pd.__version__)

## 3. Point notebooks at the repo (5 min)

In [ ]:
# This notebook lives in workshop/sessions/
# Add workshop/ to the path so `import analysis` works.
import sys
from pathlib import Path

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else SESSION_DIR / "workshop"
REPO = WORKSHOP_DIR.parent if WORKSHOP_DIR.name == "workshop" else SESSION_DIR.parents[1]

sys.path.insert(0, str(WORKSHOP_DIR))
from analysis.paths import data_path, stimuli_path, REPO_ROOT

print("REPO_ROOT =", REPO_ROOT)
print("Food data exists?", data_path("food_decision_making").exists())
print("Stimuli exist?", stimuli_path("Decision Making").exists())
list(data_path("food_decision_making").glob("*"))

## 4. Load workshop datasets (25 min)

### 4a. Tobii food metrics (event table — friendly first view)


In [ ]:
metrics = pd.read_csv(
    data_path("food_decision_making", "Food_Decision_Making_Metrics.tsv"),
    sep="\t",
)
metrics.head()

In [ ]:
print(metrics.shape)
print(metrics.columns.tolist())
metrics["AOI"].value_counts(dropna=False).head(15)

### 4b. Teaching sample (one recording, sample-level gaze + mouse)

In [ ]:
sample = pd.read_csv(data_path("food_decision_making", "Food_Decision_Making_Teaching_Sample.csv"))
print(sample.shape)
sample[["Recording timestamp", "Sensor", "Presented Stimulus name", "Gaze point X", "Gaze point Y", "Eye movement type"]].head(10)

In [ ]:
sample["Sensor"].value_counts(dropna=False)
# How many gaze samples have coordinates?
gaze = sample.query("Sensor == 'Eye Tracker'").copy()
gaze["Gaze point X"].notna().mean()

### 4c. Pupil Labs fixations & blinks

In [ ]:
pupil_fix = pd.read_csv(data_path("pupil_labs_recording", "fixations.csv"))
pupil_blinks = pd.read_csv(data_path("pupil_labs_recording", "blinks.csv"))
print("fixations", pupil_fix.shape, "blinks", pupil_blinks.shape)
pupil_fix[["start_timestamp", "duration", "norm_pos_x", "norm_pos_y", "confidence"]].head()

### 4d. Tobii GSR metrics (wide table)

In [ ]:
gsr = pd.read_csv(data_path("tobii_gsr_demo", "Tobii_Pro_Lab_GSR_Demo_Project_Metrics.tsv"), sep="\t")
print(gsr.shape)
keep = [c for c in gsr.columns if c in {
    "Recording","Participant","TOI","Media","Average_GSR","Number_of_SCR",
    "Last_key_press","Number_of_whole_fixations","Average_whole-fixation_pupil_diameter"
} or c.startswith("Number_of_mouse_clicks.") or c.startswith("Time_to_first_fixation.")]
gsr[keep].head()

## 5. Excel-brain → pandas-brain (15 min)

| Excel | pandas |
|-------|--------|
| Filter rows | `df.query("AOI == 'cake-pic'")` |
| Pivot / average | `df.groupby("AOI")["Duration"].mean()` |
| Sort | `df.sort_values("Duration", ascending=False)` |
| Save | `df.to_csv("out.csv", index=False)` |


In [ ]:
# Duration in the metrics file is often milliseconds — check a few values
m = metrics.dropna(subset=["Duration", "AOI"]).copy()
summary = (
    m.groupby("AOI", as_index=False)["Duration"]
    .agg(n="count", mean_ms="mean", median_ms="median")
    .sort_values("mean_ms", ascending=False)
)
summary.head(12)

In [ ]:
ax = summary.head(8).plot(x="AOI", y="mean_ms", kind="bar", legend=False, color="#2a6f6f")
ax.set_ylabel("Mean fixation duration (ms)")
ax.set_title("Session 1 — first real metric plot")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Mini practice (10 min)

1. From `metrics`, keep only rows where `Event_type` looks like fixations (inspect unique values).
2. Compute mean `Average_pupil_size` by `AOI`.
3. Name one analysis question you could ask with **mouse clicks + gaze** in the GSR table.

## Exit ticket
Write one sentence: *Which file will you open first in Session 2 for I-VT, and why?*
